# Shapedb Search Session

This notebook prepares a PDB-derived ligand shape file and launches a shapedb search against the Enamine database.

- It saves the ligand segment `LIG` from the PDB as `.mol2`.
- Then it converts and centers that `.mol2` to `_centered.sdf` using `func/shapedb/convert_and_center_mol2.py`.
- Finally it uses the generated `.sdf` for the shapedb search.

In [2]:
## input a pdb file with desing truncated structure
import os
from pathlib import Path
import subprocess
import sys
import pymol
from pymol import cmd
!module load lmod 

def run_cmd(cmd):
	result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
	if result.stdout:
		print(result.stdout, end="")
	if result.stderr:
		print(result.stderr, file=sys.stderr, end="")
	return result.returncode

#########
working_dir ="/pi/summer.thyme-umw/Ji_rosetta_discovery/"
enamine_database  = "/pi/summer.thyme-umw/enamine-REAL-2.6billion"
########



In [35]:
#prep ligand mol2 files and convert to centered sdf for shapedb search

pdb_path = Path(working_dir) / "input_pdb"
shapedb_output_root = Path(working_dir) / "out/shapedb"
convert_script = Path(working_dir) / "func" / "shapedb" / "convert_and_center_mol2.py"
clean_script = Path(working_dir) / "func" / "shapedb" / "clean_pdb.py"

print("working_dir:", working_dir)
print("pdb_path:", pdb_path)
print("shapedb_output_root:", shapedb_output_root)

if not pdb_path.exists():
    raise FileNotFoundError(f"PDB file not found: {pdb_path}")

for pdb_file in pdb_path.glob("*.pdb"):
    stem = pdb_file.stem
    shapedb_output = shapedb_output_root / stem
    ligand_mol2 = shapedb_output / f"{stem}.mol2"
    ligand_sdf = shapedb_output / f"{stem}_centered.sdf"

    print("Processing PDB:", pdb_file)
    shapedb_output.mkdir(parents=True, exist_ok=True)
    cmd.reinitialize()
    cmd.load(str(pdb_file), object='agonist')
    cmd.remove('chain R')
    cmd.save(str(ligand_mol2), 'segid LIG', format='mol2')
    print("Saved MOL2:", ligand_mol2)

    if not ligand_sdf.exists():
        print("Converting MOL2 to centered SDF:", ligand_mol2)
        subprocess.run(["python", str(convert_script), str(shapedb_output)], check=True)
    else:
        print("Centered SDF already exists:", ligand_sdf)

    print("Output SDF:", ligand_sdf)


for pdb_file in pdb_path.glob("*.pdb"):
    stem = pdb_file.stem
    print("Processing PDB:", pdb_file)
    cmd.reinitialize()
    cmd.load(str(pdb_file), object='RECEPTOR')
    cmd.remove('chain L')
    pdb_out = Path(pdb_path) / "processed" / (stem + "_receptor.pdb")
    cmd.save(str(pdb_out), 'chain R', format='pdb')
    os.system("python {} {} {}".format(clean_script, str(pdb_out), "R"))
    print("Saved PDB:", str(pdb_out))



working_dir: /pi/summer.thyme-umw/Ji_rosetta_discovery/
pdb_path: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb
shapedb_output_root: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb
Processing PDB: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_lemborexant.pdb
Saved MOL2: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexant/orexin_lemborexant.mol2
Centered SDF already exists: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexant/orexin_lemborexant_centered.sdf
Output SDF: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_lemborexant/orexin_lemborexant_centered.sdf
Processing PDB: /pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/orexin_state3.pdb
Saved MOL2: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3.mol2
Centered SDF already exists: /pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state3/orexin_state3_centered.sdf
Output SDF: /pi/summer.thyme-umw/Ji_rosetta_dis

In [13]:
# check your output sdf, if it's good set this to True to execute the shapedb search.
run_search = False

ligand_mol2 =  "/pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state4/orexin_state4_centered.sdf"
shapedb_output = "/pi/summer.thyme-umw/Ji_rosetta_discovery/out/shapedb/orexin_state4/"
if run_search:
    shapedb_controller = Path(working_dir) / "func" / "shapedb" / "nnsearch_controller.py"
    shapedb_cmd = ["bsub -q long -n 15 -W 24:00 -R \"rusage[mem=4000]\" -J shapedb_search_ctrl" ,
        "python",
        str(shapedb_controller),
        str(ligand_mol2),
        "--working-dir",
        str(shapedb_output),
        "--realm-dir","/pi/summer.thyme-umw/Ji_rosetta_discovery",
        "-n", "3000000", "-j", "500", "--min-chunk", "0", "--max-chunk", "53084",

    ]
    print("Executing shapedb search:", " ".join(shapedb_cmd))
    run_cmd(" ".join(shapedb_cmd))
else:
    print("Search not executed.")
    print("Set run_search = True and rerun this cell when you want to submit shapedb.")


Search not executed.
Set run_search = True and rerun this cell when you want to submit shapedb.


In [ ]:
#check your shapedb output, if it's good set this to True to execute the discovery pipeline on the top hits from shapedb.
run_discovery = True
conformator_license = "/pi/summer.thyme-umw/Ji_rosetta_discovery/license/conformator.txt"
top_ligands_list = Path("./out/shapedb/orexin_lemborexant") / "combined_results/combined_best_3000000_chunks_00000_53084.txt"
discovery_output = Path("/pi/summer.thyme-umw/Ji_rosetta_discovery/") / "out/discovery/orexin_lemborexant"
if run_discovery:
    discovery_cmd = ["bsub -q long -n 8 -W 168:00 -R \"rusage[mem=2000]\" -J discovery_ligand_ctrl" ,
        "python", 
        "./func/discovery/unified_discovery_controller.py",
         "--shapedb-list", str(top_ligands_list),
         "--target-pdb", "/pi/summer.thyme-umw/Ji_rosetta_discovery/input_pdb/processed/orexin_lemborexant_receptor_R.pdb",
         "--anchor-residues", "138", #must be generic in the input pdb 
         "--motifs-file", "/pi/summer.thyme-umw/Ji_rosetta_discovery/motifs/FINAL_motifs_list_filtered_2_3_2023.motifs", ##must be absolute path
         "--workers", "30",
         "--top-hits", "1",
         "--output-dir", str(discovery_output),
         "--conformator-license", conformator_license,
         "--extra-params", "/pi/summer.thyme-umw/Ji_rosetta_discovery/extra_arg/sample_extra_arg" ##optional extra params for rosetta, can be left out or modified as needed
    ]
    
    run_cmd(" ".join(discovery_cmd))

Job <172622> is submitted to queue <long>.


WARN: No span specified with (-n 8). Defaulting to span[hosts=1] (single host). See https://tinyurl.com/3rctk5j9


In [26]:
run_cleanup = True
top_ligands_list = Path(shapedb_output) / "combined_results/test.txt"
if run_cleanup:
    cleanup_cmd = ["bsub -q long -n 4 -W 168:00 -R \"rusage[mem=4000]\" -J param_ligand_ctrl" ,
        "python", 
        "./func/discovery_test_params_preparation/prepare_test_params_directories_controller.py",
        str(shapedb_output),
        str(top_ligands_list),
        "--workers", "4",
    ]
    
    run_cmd(" ".join(cleanup_cmd))

Job <149634> is submitted to queue <long>.


WARN: No span specified with (-n 4). Defaulting to span[hosts=1] (single host). See https://tinyurl.com/3rctk5j9
